# 10 — Schema-Driven Flatten of the COSMoS Graph

Step 2a of the cosmos-bc-dss rewrite — converts the CDISC Excel exports into a multi-sheet interim that preserves the authored graph shape at VLM-row grain. No external-data lookup, no CT resolution, no NCIt enrichment. Schema-driven via `linkml-runtime.SchemaView`.

**Input**
- `downloads/cdisc_sdtm_dataset_specializations_latest.xlsx` — 12,677 rows × 32 cols at VLM-row grain
- `downloads/cdisc_biomedical_concepts_latest.xlsx` — BC identity + hierarchy
- `cosmos-graph/reference/cosmos_linkml/cosmos_sdtm_model.yaml` — LinkML schema for SDTMGroup / SDTMVariable / RelationShip
- `cosmos-graph/reference/cosmos_linkml/cosmos_bc_model.yaml` — LinkML schema for BiomedicalConcept

**Output**
- `interim/COSMoS_Graph.xlsx` with sheets: `ReadMe`, `DSS`, `Variables`, `Relationships`, `Codelists`, `BC`

**Out of scope for this notebook** — CT resolution (→ notebook 20), Identity derivation (→ notebook 30), NCIt enrichment (→ Step 3).

Design rationale: `cosmos-bc-dss/docs/COSMoS_Flattener_Rewrite.md`.


## 1. Imports


In [ ]:
import sys
from pathlib import Path
from datetime import date

import pandas as pd
from linkml_runtime.utils.schemaview import SchemaView

print(f'python : {sys.version.split()[0]}')
print(f'pandas : {pd.__version__}')


## 2. Configuration


In [ ]:
REPO_ROOT = Path.cwd().parent.parent  # notebooks/ -> cosmos-graph/ -> repo root

DOWNLOADS = REPO_ROOT / 'cosmos-bc-dss' / 'downloads'
INTERIM   = REPO_ROOT / 'cosmos-graph' / 'interim'
SCHEMAS   = REPO_ROOT / 'cosmos-graph' / 'reference' / 'cosmos_linkml'

DSS_XLSX    = DOWNLOADS / 'cdisc_sdtm_dataset_specializations_latest.xlsx'
BC_XLSX     = DOWNLOADS / 'cdisc_biomedical_concepts_latest.xlsx'
SDTM_SCHEMA = SCHEMAS / 'cosmos_sdtm_model.yaml'
BC_SCHEMA   = SCHEMAS / 'cosmos_bc_model.yaml'
OUT_XLSX    = INTERIM / 'COSMoS_Graph.xlsx'

for p in (DSS_XLSX, BC_XLSX, SDTM_SCHEMA, BC_SCHEMA):
    assert p.exists(), f'missing: {p}'
INTERIM.mkdir(parents=True, exist_ok=True)

print(f'DSS_XLSX : {DSS_XLSX.relative_to(REPO_ROOT)}')
print(f'BC_XLSX  : {BC_XLSX.relative_to(REPO_ROOT)}')
print(f'out      : {OUT_XLSX.relative_to(REPO_ROOT)}')


## 3. Load LinkML schemas

The schemas are the authoritative model — we use SchemaView to enumerate slots and drive column selection. This notebook does not attempt structural validation of the xlsx against the schema (notebook 01 already confirmed shape equivalence for ABI); we rely on the schema as a slot catalogue.


In [ ]:
sv_sdtm = SchemaView(str(SDTM_SCHEMA))
sv_bc   = SchemaView(str(BC_SCHEMA))

sdtm_classes = list(sv_sdtm.all_classes())
bc_classes   = list(sv_bc.all_classes())
print(f'sdtm schema: {len(sdtm_classes)} classes')
print(f'bc schema  : {len(bc_classes)} classes')

sdtm_var_slots = sv_sdtm.class_induced_slots('SDTMVariable')
print(f'\nSDTMVariable: {len(sdtm_var_slots)} induced slots')
print('  ' + ', '.join(s.name for s in sdtm_var_slots[:12]) + ' ...')


## 4. Read source xlsx


In [ ]:
dss_df = pd.read_excel(DSS_XLSX, sheet_name='SDTM Dataset Specializations')
print(f'DSS xlsx (VLM-row grain): {dss_df.shape}')

bc_concepts  = pd.read_excel(BC_XLSX, sheet_name='Biomedical Concepts')
bc_hierarchy = pd.read_excel(BC_XLSX, sheet_name='BC Hierarchy')
print(f'BC Biomedical Concepts : {bc_concepts.shape}  (one row per BC per external-code mapping)')
print(f'BC Hierarchy           : {bc_hierarchy.shape}  (one row per BC)')


## 5. Column rename — xlsx surface → canonical snake_case

The xlsx uses CDISC's publication names (`vlm_group_id`, `vlm_source`, `bc_id`, etc.). We rename to the canonical snake_case vocabulary used across `interim/COSMoS_Graph.xlsx` sheets. Names were agreed in `COSMoS_Flattener_Rewrite.md` (Group A table) and correspond one-to-one to LinkML slot names in snake_case form.


In [ ]:
DSS_RENAME = {
    'package_date':               'package_date',
    'bc_id':                      'bc_id',
    'sdtmig_start_version':       'sdtmig_start_version',
    'sdtmig_end_version':         'sdtmig_end_version',
    'domain':                     'domain',
    'vlm_source':                 'source',
    'vlm_group_id':               'ds_id',
    'short_name':                 'ds_short_name',
    'sdtm_variable':              'variable_name',
    'dec_id':                     'dec_id',
    'nsv_flag':                   'is_nonstandard',
    'codelist':                   'codelist_concept_id',
    'codelist_submission_value':  'codelist_submission_value',
    'subset_codelist':            'subset_codelist',
    'value_list':                 'value_list',
    'assigned_term':              'assigned_term_concept_id',
    'assigned_value':             'assigned_term_value',
    'role':                       'role',
    'subject':                    'subject',
    'linking_phrase':             'linking_phrase',
    'predicate_term':             'predicate_term',
    'object':                     'object',
    'data_type':                  'data_type',
    'length':                     'length',
    'format':                     'format',
    'significant_digits':         'significant_digits',
    'mandatory_variable':         'mandatory_variable',
    'mandatory_value':            'mandatory_value',
    'origin_type':                'origin_type',
    'origin_source':              'origin_source',
    'comparator':                 'comparator',
    'vlm_target':                 'vlm_target',
}

missing = set(dss_df.columns) - set(DSS_RENAME.keys())
extra   = set(DSS_RENAME.keys()) - set(dss_df.columns)
assert not missing and not extra, f'DSS_RENAME mismatch. missing={missing}, extra={extra}'

df = dss_df.rename(columns=DSS_RENAME).copy()
print(f'renamed: {df.shape}')
print(f'columns: {list(df.columns)}')


## 6. Build `DSS` sheet

One row per Dataset Specialization. DSS-level facts only — fields that are constant across all variable rows of a given `ds_id`. Per-variable facts live in `Variables`.


In [ ]:
DSS_COLS = ['ds_id', 'bc_id', 'domain', 'source', 'ds_short_name',
            'sdtmig_start_version', 'sdtmig_end_version', 'package_date']

dss_sheet = df[DSS_COLS].drop_duplicates(subset=['ds_id']).reset_index(drop=True)
assert dss_sheet['ds_id'].is_unique, 'ds_id not unique after dedup'

# Integrity check — no DSS-level field should vary within a ds_id group.
per_dss_variance = df.groupby('ds_id')[DSS_COLS].nunique().max()
varying = per_dss_variance[per_dss_variance > 1]
assert varying.empty, f'DSS-level columns vary within ds_id: {varying.to_dict()}'

print(f'DSS sheet: {dss_sheet.shape}')
print(f'distinct domains: {dss_sheet.domain.nunique()}')
dss_sheet.head()


## 7. Build `Variables` sheet

One row per SDTMVariable within a DSS — VLM-row grain. DSS-level columns dropped (they're in the `DSS` sheet, joined via `ds_id`). `bc_id` is kept as a denormalised join key to `BC` for convenience.


In [ ]:
VARIABLES_COLS = [
    'ds_id', 'bc_id',                                              # join keys
    'variable_name', 'role', 'is_nonstandard', 'dec_id',           # variable identity
    'data_type', 'length', 'format', 'significant_digits',          # data shape
    'mandatory_variable', 'mandatory_value', 'comparator',          # pin vs qualifier
    'origin_type', 'origin_source', 'vlm_target',                   # provenance
    'assigned_term_concept_id', 'assigned_term_value',              # pinned term
    'codelist_concept_id', 'codelist_submission_value',             # codelist binding
    'subset_codelist', 'value_list',                                # constrained values
    'subject', 'linking_phrase', 'predicate_term', 'object',        # reification quad
]

variables_sheet = df[VARIABLES_COLS].reset_index(drop=True)
print(f'Variables sheet: {variables_sheet.shape}')
print(f'role distribution: {dict(variables_sheet.role.value_counts())}')
variables_sheet.head()


## 8. Build `Relationships` sheet

Long-format reification quads. One row per reified edge. Variable rows without a populated quad are excluded — tracked as an anomaly.


In [ ]:
rel_mask = (
    variables_sheet['subject'].notna()
    & variables_sheet['linking_phrase'].notna()
    & variables_sheet['predicate_term'].notna()
    & variables_sheet['object'].notna()
)
relationships_sheet = variables_sheet.loc[rel_mask, [
    'ds_id', 'variable_name', 'subject', 'linking_phrase', 'predicate_term', 'object',
]].reset_index(drop=True)

dropped = (~rel_mask).sum()
print(f'Relationships sheet: {relationships_sheet.shape}')
print(f'dropped {dropped} variable rows with empty reification quad')
relationships_sheet.head()


## 9. Build `Codelists` sheet

Deduped codelist bindings with usage counts. One row per `(codelist_concept_id, codelist_submission_value)` pair actually referenced by any variable. `subset_codelist` is out of scope for this sheet — it stays in `Variables` as authored (range: string).


In [ ]:
codelists_sheet = (
    variables_sheet[['codelist_concept_id', 'codelist_submission_value']]
    .dropna(subset=['codelist_concept_id'])
    .value_counts(dropna=False)
    .reset_index(name='variable_uses_count')
    .sort_values('codelist_concept_id')
    .reset_index(drop=True)
)
print(f'Codelists sheet: {codelists_sheet.shape}')
codelists_sheet.head(10)


## 10. Build `BC` sheet

One row per Biomedical Concept — authored identity, hierarchy info, plus our `bc_type` classification:

- `full` — BC is in the Biomedical Concepts sheet and has at least one DSS.
- `full_no_ds` — BC is in Biomedical Concepts but no DSS has been authored (gap marker).
- `hierarchy_only` — BC is only in the BC Hierarchy sheet (structural parent, not a standalone concept).

`ncit_code` is joined from the Biomedical Concepts sheet; `bc_parent_label` is derived by mapping `parent_bc_id` back to the BC's `bc_short_name`.


In [ ]:
# BC identity — start from BC Hierarchy (authoritative BC list, one row per BC)
bc_base = bc_hierarchy[[
    'bc_id', 'short_name', 'parent_bc_id', 'bc_categories', 'synonyms',
    'result_scales', 'definition', 'bc_hierarchy_full',
]].rename(columns={
    'short_name':        'bc_short_name',
    'definition':        'bc_definition',
    'synonyms':          'bc_synonyms',
    'bc_hierarchy_full': 'bc_hierarchy_path',
})

# Join ncit_code from Biomedical Concepts (first row per bc_id — ncit_code is stable within a BC)
bc_ncit = (
    bc_concepts[['bc_id', 'ncit_code']]
    .dropna(subset=['bc_id'])
    .drop_duplicates(subset=['bc_id'])
)
bc_sheet = bc_base.merge(bc_ncit, on='bc_id', how='left')

# Parent label — map parent_bc_id -> bc_short_name
name_map = dict(zip(bc_sheet['bc_id'], bc_sheet['bc_short_name']))
bc_sheet['bc_parent_label'] = bc_sheet['parent_bc_id'].map(name_map)

# bc_type classification
bcs_in_concepts = set(bc_concepts['bc_id'].dropna().unique())
bcs_with_dss    = set(dss_sheet['bc_id'].dropna().unique())

def classify_bc(bc_id):
    if bc_id not in bcs_in_concepts:
        return 'hierarchy_only'
    if bc_id in bcs_with_dss:
        return 'full'
    return 'full_no_ds'

bc_sheet['bc_type'] = bc_sheet['bc_id'].apply(classify_bc)

BC_COLS = [
    'bc_id', 'ncit_code', 'bc_short_name', 'bc_definition', 'bc_synonyms',
    'bc_categories', 'bc_hierarchy_path', 'bc_parent_label', 'bc_type',
    'result_scales',
]
bc_sheet = bc_sheet[BC_COLS].reset_index(drop=True)

print(f'BC sheet: {bc_sheet.shape}')
print(f'bc_type distribution:')
print(bc_sheet['bc_type'].value_counts())
bc_sheet.head()


## 11. Anomaly audit

Counts for issues previously surfaced by notebook 02 and any new ones. These go into the ReadMe; the flattener does not silently fix anything.


In [ ]:
dss_with_edges = set(relationships_sheet['ds_id'].unique())
dss_all        = set(dss_sheet['ds_id'].unique())

issues = {
    'vlm_source_hyphen_rows':
        int(dss_df['vlm_source'].str.contains('-', na=False).sum()),
    'empty_reification_quad_rows':
        int((~rel_mask).sum()),
    'dss_without_any_edge':
        len(dss_all - dss_with_edges),
    'codelist_null_rows':
        int(variables_sheet['codelist_concept_id'].isna().sum()),
    'assigned_term_null_rows':
        int(variables_sheet['assigned_term_concept_id'].isna().sum()),
}
for k, v in issues.items():
    print(f'  {k:35s} : {v:>6,}')


## 12. Write `interim/COSMoS_Graph.xlsx`

Five data sheets + ReadMe. Identity derivation happens in notebook 30, CT resolution in notebook 20.


In [ ]:
readme_lines = [
    'COSMoS Graph — schema-driven flatten of CDISC Library Excel exports',
    '',
    f'Generated : {date.today().isoformat()}',
    'Pipeline  : cosmos-graph/notebooks/10_flatten_schema_driven.ipynb',
    f'Input     : {DSS_XLSX.name}, {BC_XLSX.name}',
    'Schema    : cosmos-graph/reference/cosmos_linkml/ (cosmos_sdtm_model.yaml, cosmos_bc_model.yaml)',
    '',
    'SHEETS',
    f'  DSS            {len(dss_sheet):>6,} rows — one row per Dataset Specialization',
    f'  Variables      {len(variables_sheet):>6,} rows — one row per SDTMVariable (VLM-row grain)',
    f'  Relationships  {len(relationships_sheet):>6,} rows — reification quads',
    f'  Codelists      {len(codelists_sheet):>6,} rows — deduped codelist bindings with usage counts',
    f'  BC             {len(bc_sheet):>6,} rows — one row per Biomedical Concept',
    '',
    'ANOMALIES',
]
for k, v in issues.items():
    readme_lines.append(f'  {k:35s} : {v:,}')
readme_lines += [
    '',
    'See cosmos-bc-dss/docs/COSMoS_Flattener_Rewrite.md for design rationale.',
    'See cosmos-bc-dss/docs/COSMoS_Graph_As_Authored.md for the authored-graph model.',
]

readme_df = pd.DataFrame({'README': readme_lines})

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as writer:
    readme_df.to_excel(writer, sheet_name='ReadMe', index=False, header=False)
    dss_sheet.to_excel(writer, sheet_name='DSS', index=False)
    variables_sheet.to_excel(writer, sheet_name='Variables', index=False)
    relationships_sheet.to_excel(writer, sheet_name='Relationships', index=False)
    codelists_sheet.to_excel(writer, sheet_name='Codelists', index=False)
    bc_sheet.to_excel(writer, sheet_name='BC', index=False)

print(f'wrote {OUT_XLSX}')
print(f'size : {OUT_XLSX.stat().st_size:,} bytes')


## 13. Summary


In [ ]:
xl = pd.ExcelFile(OUT_XLSX)
print(f'sheets in output: {xl.sheet_names}')
for s in xl.sheet_names:
    sh = pd.read_excel(OUT_XLSX, sheet_name=s, header=None if s == 'ReadMe' else 0)
    print(f'  {s:15s}: {sh.shape}')
